# Collecting Human Preferences for post-training with the Prolific AI Task Builder

The **Prolific AI Task Builder** provides a streamlined way to:
- Present multiple model responses to participants
- Collect reliable human preferences at scale
- Export results in a standardized format ready for machine learning workflows

## What This Notebook Does

**Collect Human Preferences** 
- Upload response pairs to Prolific
- Create and publish a study for human annotators
- Wait for participants to complete the tasks
- Download and process the results

**Output:** A preference tuning dataset in `(prompt, chosen_response, rejected_response)` format ready for reward model training.

---

## Setup

First, we'll import the necessary libraries and our custom modules.

In [1]:
import sys, yaml, json
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import datetime

sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Import our custom modules
from prolific_ai_taskers import (
    ProlificClient,             # For interacting with Prolific API
    process_preferences         # For processing Prolific responses into RLHF format
)

## Configuration

Load environment variables (API tokens) and configuration settings.

In [27]:
# Load environment variables from .env file
# This file should contain:
#   - PROLIFIC_API_TOKEN (Prolific API token)
#   - PROLIFIC_WORKSPACE_ID (your Prolific workspace)
#   - PROLIFIC_PROJECT_ID (your Prolific project)
load_dotenv()

# Set up file paths
config_path = Path('../config.yaml')
output_dir = Path('output_examples')
output_dir.mkdir(exist_ok=True)

print("✅ Paths configured")

✅ Paths configured


In [28]:
# Load configuration from YAML file
# This contains Prolific study parameters
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

---

# Collect Human Preferences via Prolific

Now we'll upload our response pairs to Prolific and collect human preferences.

**Prerequisites:**
- Prolific account with API access
- Valid API token, workspace ID, and project ID in your `.env` file
- Sufficient funds in your Prolific account to pay participants

**What happens in this section:**
1. Create a dataset and upload response pairs
2. Configure the annotation task (how participants will see the data) using Prolific's AI Task Builder
3. Create and publish a study
4. Wait for participants to complete the tasks
5. Download and process the results

## Step 1: Initialize Prolific Client

This authenticates with Prolific using your credentials from the `.env` file.

In [4]:
# Initialize Prolific client
# This will read credentials from environment variables and authenticate
client = ProlificClient()

✅ Authenticated as Viviana Marquez


In [5]:
pairs_csv_path = Path("input_examples/response_pairs.csv")
pairs_csv = pd.read_csv(pairs_csv_path)
pairs_csv.head()

,Prompt,Response A,Response B
0,What are the main differences between Python a...,It is a high-level programming language with d...,What are some of the benefits of using each la...
1,What are the main differences between Python a...,It is a high-level programming language with d...,How to convert one to another? Is it possible ...
2,What are the main differences between Python a...,It is a high-level programming language with d...,Let’s take a look at the most prominent differ...
3,What are the main differences between Python a...,What are some of the benefits of using each la...,How to convert one to another? Is it possible ...
4,What are the main differences between Python a...,What are some of the benefits of using each la...,Let’s take a look at the most prominent differ...


## Step 2: Create Dataset and Upload Data

A "dataset" in Prolific is a container for your data. We'll:
1. Create a new dataset
2. Upload the response pairs CSV
3. Wait for Prolific to process it

In [6]:
# Create a dataset
dataset_name = cfg['prolific']['batch_name']
dataset_id = client.create_dataset(name=dataset_name)

✅ Created dataset: 019c21ff-df68-7013-8c1d-6f1dfdf4e9dd


In [7]:
# Upload the response pairs CSV
client.upload_dataset_file(dataset_id, pairs_csv_path)

✅ Uploaded response_pairs.csv


In [8]:
# Wait for Prolific to process the dataset 
if not client.wait_for_dataset_ready(dataset_id):
    raise Exception("Dataset processing failed. Check your CSV format.")

Dataset status: UNINITIALISED
Dataset status: READY
✅ Dataset ready


## Step 3: Define Task Schema

The task schema defines what participants will see and how they'll respond.

For pairwise preference collection, we show:
- The prompt
- Response A
- Response B
- A choice: "Which response is better?"

In [9]:
# Define how the task will look to participants
task_details = {
    'task_name': cfg['prolific']['task_schema']['task_name'],
    'task_introduction': cfg['prolific']['task_schema']['task_introduction'].replace('\n', ' '),
    'task_steps': cfg['prolific']['task_schema']['task_steps'].replace('\n', ' '),
    
    # Map CSV columns to display labels
    'inputs': [
        {'key': 'prompt', 'label': 'Prompt'},
        {'key': 'response_a', 'label': 'Response A'},
        {'key': 'response_b', 'label': 'Response B'},
    ],
    
    # Define the question participants will answer
    'judgment': {
        'type': 'single_choice',
        'options': [
            {'value': 'A', 'label': 'Choose Response A'},
            {'value': 'B', 'label': 'Choose Response B'},
        ]
    },
    
    # Randomize which response shows as A or B to avoid position bias
    'randomize_inputs': ['Response A', 'Response B'],
    
    # Require a choice (no skipping)
    'validation': {'require_choice': True}
}

print("✅ Task schema defined")

✅ Task schema defined


## Step 4: Create and Configure Batch

A "batch" in Prolific groups your tasks together and defines how they're presented.

In [10]:
# Create batch
batch_id = client.create_batch(
    name=cfg['prolific']['batch_name'],
    dataset_id=dataset_id,
    task_details=task_details
)

✅ Created batch: 019c21ff-ff5a-76ed-b205-d777c645d694


In [11]:
# Add instructions for participants
instructions = [{
    'type': 'multiple_choice',
    'created_by': client.researcher_name,
    'description': cfg['prolific']['task_schema']['task_question'],
    'options': [
        {'label': 'Response A is better', 'value': 'A'},
        {'label': 'Response B is better', 'value': 'B'},
    ]
}]

client.add_batch_instructions(batch_id, instructions)

✅ Added instructions


In [12]:
# Initialize the batch
# tasks_per_group determines how many comparisons each participant does
client.initialize_batch(
    batch_id=batch_id,
    dataset_id=dataset_id,
    tasks_per_group=cfg['prolific']['task_schema']['tasks_per_group']
)

✅ Initialized batch


In [13]:
# Wait for batch to be ready (Prolific processes it in the background)
if not client.wait_for_batch_ready(batch_id):
    raise Exception("Batch setup failed. Check your task configuration.")

Batch status: READY
✅ Batch ready


<p style="text-align:center; margin-bottom:4px;">
  <img src="img/01_batch.png" alt="Batch 1" width="45%" style="border:2px solid #007BFF; border-radius:8px; margin-right:5px;">
  <img src="img/02_batch.png" alt="Batch 2" width="45%" style="border:2px solid #007BFF; border-radius:8px; margin-left:5px;">
</p>

<p style="text-align:center; font-style:italic; color:#555; margin-top:0; font-size:0.9em;">
  Preview of Prolific AI Task Builder interface after completing Steps 2.2-2.4. Left: List of created batches. Right: Batch configuration showing the task template with sample data and instructions.
</p>


## Step 5: Create and Publish Study

A "study" is what participants see and sign up for. It includes:
- Payment amount
- Estimated time
- Participant eligibility filters
- The link to your batch

In [29]:
timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M")

# AI Task Builder config
batch_name = f"{timestamp}_{cfg['prolific']['batch_name']}"
task_name = cfg['prolific']['task_schema']['task_name']
task_question = cfg['prolific']['task_schema']['task_question']
tasks_per_group = cfg['prolific']['task_schema']['tasks_per_group']  # how many comparisons each participant will do per assignment
task_introduction = cfg['prolific']['task_schema']['task_introduction']
task_steps = cfg['prolific']['task_schema']['task_steps']

# Study config 
internal_name = batch_name
estimated_completion_time = cfg['prolific']['study_setup']['estimated_completion_time']  # minutes per participant
max_time = cfg['prolific']['study_setup']['max_time']  # minutes per participant
reward = cfg['prolific']['study_setup']['reward'] # payout per participant (in cents in the currency of your account)
device_compatibility = cfg['prolific']['study_setup']['device_compatibility']
participants_per_task = cfg['prolific']['study_setup']['participants_per_task']
dc_method = cfg['prolific']['study_setup']['data_collection_method']
study_labels = cfg['prolific']['study_setup']['labels']

# Eligibility
elig = cfg["prolific"]["study_setup"].get("filters", {})
age_cfg = elig.get("age")
country_choice_ids = elig.get("current_country_of_residence_choice_ids", [])


# Create study
study_id = client.create_study(
    task_name=task_name,
    internal_name=internal_name,
    description=task_introduction,
    data_collection_method=dc_method,
    batch_id=batch_id,
    participants_per_task=participants_per_task,
    study_labels=study_labels,
    estimated_completion_time=estimated_completion_time,
    max_time=max_time,
    reward=reward,
    device_compatibility=device_compatibility,
    age_cfg=age_cfg,
    country_choice_ids=country_choice_ids
)

print(f"\n📊 Study created: {study_id}")
print(f"View in Prolific dashboard: https://app.prolific.com/researcher/workspaces/studies/{study_id}")

✅ Created study: 69818aa23095550814b7dd11

📊 Study created: 69818aa23095550814b7dd11
View in Prolific dashboard: https://app.prolific.com/researcher/workspaces/studies/69818aa23095550814b7dd11


In [30]:
# Publish the study!
# After this, participants can see and sign up for your study
client.publish_study(study_id)

print("\n🎉 Study is now live!")
print("Participants can start working on it.")

✅ Published study

🎉 Study is now live!
Participants can start working on it.


<p style="text-align:center; margin-bottom:4px;">
  <img src="img/03_study.png" alt="Batch 1" width="45%" style="border:2px solid #007BFF; border-radius:8px; margin-right:5px;">
</p>

<p style="text-align:center; font-style:italic; color:#555; margin-top:0; font-size:0.9em;">
 Creator view in Prolific after publishing the study (Step 2.5). Your published study appears in the studies list with its status, participant allocation, and study details.
</p>


<p style="text-align:center; margin-bottom:4px;">
  <img src="img/04_participant.png" alt="Batch 1" width="45%" style="border:2px solid #007BFF; border-radius:8px; margin-right:5px;">
  <img src="img/05_participant.png" alt="Batch 2" width="45%" style="border:2px solid #007BFF; border-radius:8px; margin-left:5px;">
</p>

<p style="text-align:center; margin-bottom:4px;">
  <img src="img/06_participant.png" alt="Batch 1" width="45%" style="border:2px solid #007BFF; border-radius:8px; margin-right:5px;">
  <img src="img/07_participant.png" alt="Batch 2" width="45%" style="border:2px solid #007BFF; border-radius:8px; margin-left:5px;">
</p>

<p style="text-align:center; font-style:italic; color:#555; margin-top:0; font-size:0.9em;">
  What participants see on their end (clockwise from top-left): <br>1) What annotators see when they accept your study, 2) Detailed task instructions and criteria, 3-4) Individual comparison tasks showing a prompt with Response A and Response B, where they select the better response.
</p>

## Step 6: Wait for Study Completion

Now we wait for participants to complete the study. 

**You can:**
- Let this cell run (it polls every 60 seconds)
- Stop the notebook and come back later (save the `study_id` and `batch_id`)
- Check progress on the Prolific dashboard

**Note:** The default timeout is 6 hours. You can change it if needed.

In [ ]:
# Wait for all participants to complete the study
# This will print status updates every minute
if not client.wait_for_study_completion(study_id, timeout_sec=21600):  # 6 hours
    print("⚠️ Study didn't complete in time. Check Prolific dashboard.")
    print(f"Study ID: {study_id}")
    print(f"Batch ID: {batch_id}")
    print("\nYou can resume later by running the next cells with these IDs.")

## Step 7: Fetch and Process Results

Once the study is complete, we'll:
1. Download raw responses from all participants
2. Aggregate votes (count how many chose A vs B for each pair)
3. Apply majority voting to determine which response is preferred
4. Create the final RLHF dataset

In [55]:
# Fetch responses from Prolific
df_responses = client.fetch_batch_responses(batch_id)

# Save raw responses
raw_responses_path = output_dir / 'raw_preferences.csv'
df_responses.to_csv(raw_responses_path, index=False)
print(f"Saved raw responses → {raw_responses_path}")

✅ Fetched 14 responses
Saved raw responses → output_examples/raw_preferences.csv


In [56]:
# Preview raw responses - each row has responses from all participants
df_responses.head(3)

,DataPoint_ID,Task_Group_ID,Task_Type,Question,Prompt,Response A,Response B,Annotator1_ID,Annotator1_Response,Annotator1_Timestamp,Annotator2_ID,Annotator2_Response,Annotator2_Timestamp
0,019c21ff-ec11-7474-ac06-824451ec858a,b9e8d6b7-3b06-5b03-9793-33ad40821f4d,multiple_choice,"Pick the response that feels overall better, e...",How do I troubleshoot a slow computer?,If you notice that your computer is running sl...,What are the common culprits? Letâs look at ...,66594968a8fca6ea85c9ac19,Response A is better,2026-02-03T05:47:24.483Z,62fdd0e557fbc97b30b32ae3,Response A is better,2026-02-03T05:55:10.397Z
1,019c21ff-e9b0-7678-b127-8ff987b28005,242dc69f-5b1c-51bf-a405-1e1b58576519,multiple_choice,"Pick the response that feels overall better, e...",How can I create a budget and stick to it?,- 3 steps\nHow can I create a budget and stick...,How do I find time to get things done?\nI want...,671d5a22ce1ff61ee45332f8,Response B is better,2026-02-03T05:40:21.546Z,6966d3b7b50cd88e82df2513,Response A is better,2026-02-03T05:55:14.620Z
2,019c21ff-ec38-7399-9358-2ac9ecefe400,b9e8d6b7-3b06-5b03-9793-33ad40821f4d,multiple_choice,"Pick the response that feels overall better, e...",How do I troubleshoot a slow computer?,There are many ways to speed up a computer tha...,What are the common culprits? Letâs look at ...,66594968a8fca6ea85c9ac19,Response A is better,2026-02-03T05:47:24.483Z,62fdd0e557fbc97b30b32ae3,Response B is better,2026-02-03T05:55:10.397Z


In [57]:
# Process preferences into preference tuning format
# This function:
#   1. Counts votes for each response pair
#   2. Applies majority voting to determine chosen vs rejected
#   3. Saves both the vote counts and final RLHF dataset

df_votes = process_preferences(
    responses_df=df_responses,
    participants_per_task=cfg['prolific']['study_setup']['participants_per_task'],
    output_dir=output_dir
)

Saved votes → output_examples/votes_preferences.csv
✅ Created RLHF dataset: 6 pairs (8 ties excluded)
Saved preferences → output_examples/preferences.jsonl


In [58]:
# Look at the vote aggregation
df_votes.head(3)

,Prompt,Response A,Response B,A_wins,B_wins
0,How do I troubleshoot a slow computer?,If you notice that your computer is running sl...,What are the common culprits? Letâs look at ...,2,0
1,How can I create a budget and stick to it?,- 3 steps\nHow can I create a budget and stick...,How do I find time to get things done?\nI want...,1,1
2,How do I troubleshoot a slow computer?,There are many ways to speed up a computer tha...,What are the common culprits? Letâs look at ...,1,1


In [59]:
# Load the preference tuning dataset
with open(output_dir / 'preferences.jsonl', 'r') as f:
    rlhf_data = [json.loads(line) for line in f]

# Preview the preference tuning dataset
max_chars = 30
truncated_data = []
for item in rlhf_data[:3]:
    truncated_item = item.copy()
    for field in ["chosen_response", "rejected_response"]:
        text = truncated_item[field]
        if len(text) > max_chars:
            truncated_item[field] = text[:max_chars] + "..."
    truncated_data.append(truncated_item)

print(json.dumps(truncated_data, indent=2))

[
  {
    "prompt": "How do I troubleshoot a slow computer?",
    "chosen_response": "If you notice that your comput...",
    "rejected_response": "What are the common culprits? ..."
  },
  {
    "prompt": "How can I create a budget and stick to it?",
    "chosen_response": "4 tips for successful budgetin...",
    "rejected_response": "- 3 steps\nHow can I create a b..."
  },
  {
    "prompt": "What are the main differences between Python and JavaScript?",
    "chosen_response": "It is a high-level programming...",
    "rejected_response": "Let\u00e2\u0080\u0099s take a look at the mos..."
  }
]


## Step 8: Fetch Demographics (Optional)

Prolific provides demographic information about participants. This is useful for:
- Understanding your annotator pool
- Analyzing potential biases
- Reporting in research papers

In [60]:
# Fetch participant demographics
df_demographics = client.fetch_study_demographics(study_id)

# Save demographics
demo_path = output_dir / 'demographic.csv'
df_demographics.to_csv(demo_path, index=False)
print(f"Saved demographics → {demo_path}")

✅ Fetched demographics for 3 participants
Saved demographics → output_examples/demographic.csv


In [61]:
df_demographics.head()

,Submission id,Age,Archived at,Authenticity check results,Completed at,Completion code,Country of birth,Country of residence,Custom study tncs accepted at,Employment status,...,Nationality,Participant id,Reviewed at,Sex,Started at,Status,Student status,Time taken,Total approvals,URL
0,69818adc5c84f5f43098b2f8,37,2026-02-03T05:55:16.714184Z,0 out of 0 checks failed,2026-02-03T05:55:16.303000Z,NOCODE,Canada,United States,Not Applicable,"Not in paid work (e.g. homemaker', 'retired or...",...,United States,6966d3b7b50cd88e82df2513,NaN,Female,2026-02-03T05:42:59.121000Z,AWAITING REVIEW,No,738,123,https://app.prolific.com/data-collection-tool/...
2,69818b530bd4baef849081f6,28,2026-02-03T05:47:25.557585Z,0 out of 0 checks failed,2026-02-03T05:47:25.213000Z,NOCODE,United States,United States,Not Applicable,DATA_EXPIRED,...,United States,66594968a8fca6ea85c9ac19,NaN,Male,2026-02-03T05:44:55.149000Z,AWAITING REVIEW,DATA_EXPIRED,151,4369,https://app.prolific.com/data-collection-tool/...
5,69818cda22105a8649533abe,38,2026-02-03T05:55:11.731809Z,0 out of 0 checks failed,2026-02-03T05:55:11.378000Z,NOCODE,United States,United States,Not Applicable,DATA_EXPIRED,...,United States,62fdd0e557fbc97b30b32ae3,NaN,Male,2026-02-03T05:51:44.647000Z,AWAITING REVIEW,DATA_EXPIRED,207,3473,https://app.prolific.com/data-collection-tool/...
